In [2]:
# 修改model_weights下的文件夹名称
# run_RL_transfer.py中中查找训练好的模型文件路径为
# load_network(f'round_{round_num}_inter_0', file_path=f"./model_weights/{dic_agent_conf['MODEL_NAME']}/{traffic_file_source}/")

# 需要将model_weights下的文件夹名称修改为对应的数据集名称，例如`LLMTSCS/model_weights/AdvancedColight/synthetic_8000_1h.json01_27_14_43_35`修改为`LLMTSCS/model_weights/AdvancedColight/synthetic_8000_1h`

import os
from pathlib import Path

folder = "/home/wen/tr/LLMTSCS/model_weights"

for model_name in os.listdir(folder):
    model_path = Path(folder) / model_name
    if not model_path.is_dir():
        continue
    for dataset_name in os.listdir(model_path):
        dataset_path = model_path / dataset_name
        if not dataset_path.is_dir():
            continue
        dataset_new_name = dataset_name.split(".")[0]
        dataset_new_path = model_path / dataset_new_name
        dataset_path.rename(dataset_new_path)

# 数据结果收集

基于RL的方法在评测之后进行数据收集和整理，

In [5]:
_TRAFFIC_FILE_MAP = {
    "anon_3_4_jinan_real": "jinan1",
    "anon_3_4_jinan_real_2000": "jinan2",
    "anon_3_4_jinan_real_2500": "jinan3",
    "anon_4_4_hangzhou_real": "hangzhou1",
    "anon_4_4_hangzhou_real_5816": "hangzhou2",
    "synthetic_8000_1h": "synthetic",
}


# 保留两位小数
def round_metrics(metrics):
    return {k: round(v, 2) for k, v in metrics.items()}


import ast
import json
from pathlib import Path

log_path = "/home/wen/tr/LLMTSCS/logs/test_jinan_20260130_134704"

log_files = list(Path(log_path).glob("*.log"))

exp_res = {}

for log_file in log_files:
    file_name = log_file.name
    method = file_name.split("_")[0]
    if method == log_file.name:
        continue
    traffic_file_key = file_name.replace(f"{method}_", "").replace(".log", "")
    if traffic_file_key.startswith(method):
        continue

    with open(log_file) as f:
        lines = f.readlines()
    metrics = None
    for line in lines:
        if "{'test_reward_over'" in line:
            metrics = ast.literal_eval(line.strip())
            if type(metrics) is not dict:
                print(f"评估指标格式错误，跳过 {file_name}")
            break
    if metrics is None:
        print(f"未找到评估指标，跳过 {file_name}")
        continue
    if method not in exp_res:
        exp_res[method] = {}
    exp_res[method][_TRAFFIC_FILE_MAP.get(traffic_file_key, traffic_file_key)] = round_metrics(
        {
            "ATT": metrics["test_my_metrics"]["avg_travel_time"],
            "AWT": metrics["test_my_metrics"]["avg_waiting_time"],
            "AQL": metrics["test_my_metrics"]["avg_queue_length"],
            "EBR": metrics["test_my_metrics"]["emergency_braking_rate"],
        }
    )

with open(f"{log_path}/summary_results.json", "w") as f:
    json.dump(exp_res, f, indent=4)

"""
to csv
, jinan1_ATT,jinan1_AWT,jinan1_AQL,jinan1_EBR, jinan2, jinan3, hangzhou1, hangzhou2, synthetic
max_pressure, 23.5, 25.3, 22.1, 30.2, 28.7, 19.4
fixed_time, 28.4, 30.1, 27.5, 35.0, 33
EffectiveQL, 20.3, 21.5, 19.8, 26.4, 24.9, 18.2
"""

'\nto csv\n, jinan1_ATT,jinan1_AWT,jinan1_AQL,jinan1_EBR, jinan2, jinan3, hangzhou1, hangzhou2, synthetic\nmax_pressure, 23.5, 25.3, 22.1, 30.2, 28.7, 19.4\nfixed_time, 28.4, 30.1, 27.5, 35.0, 33\nEffectiveQL, 20.3, 21.5, 19.8, 26.4, 24.9, 18.2\n'

In [6]:
# Convert Data to CSV
import csv

metrics = ["ATT", "AWT", "AQL", "EBR"]
method_results = json.load(open(f"{log_path}/summary_results.json"))
methods = list(method_results.keys())
traffic_files = list(next(iter(method_results.values())).keys())
traffic_files.sort()

with open(f"{log_path}/summary_results.csv", "w", newline="") as csvfile:
    csvwriter = csv.writer(csvfile)
    header = [""] + [f"{tf}_{metric}" for tf in traffic_files for metric in metrics]
    csvwriter.writerow(header)

    for method in methods:
        row = [method]
        for tf in traffic_files:
            for metric in metrics:
                row.append(method_results[method][tf][metric])
        csvwriter.writerow(row)

In [ ]:
from pathlib import Path

p = Path("/home/wen/tr/rltsc/logs/async_20260127-154732/hangzhou_synthetic_trafficr1-15_qwen253_288/archives")

l = p.glob("[0-9]+")

len(list(l))